In [1]:
from pathlib import Path

import numpy as np
from numba import njit
from scipy import stats
from tqdm import tqdm

import jitr

In [2]:
import matplotlib

matplotlib.use("pgf")
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.lines import Line2D

colors = [
    "#17becf",
    "#bcbd22",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
]


rcParams["legend.fontsize"] = 14
rcParams["font.size"] = 14
rcParams["font.weight"] = "normal"
rcParams["xtick.labelsize"] = 14.0
rcParams["ytick.labelsize"] = 14.0
rcParams["lines.linewidth"] = 2.0
rcParams["xtick.major.pad"] = "10"
rcParams["ytick.major.pad"] = "10"

In [3]:
#  elastic reaction
target = (40, 20)
proton = (1, 1)
neutron = (1, 0)
projectile = neutron
Elab = 21
a = 5 * np.pi
sgrid = np.linspace(0.01, a, 100)
# for plotting differential xs
angles = np.linspace(0.1, np.pi, 100)

In [4]:
from jitr.optical_potentials.potential_forms import (
    coulomb_charged_sphere,
    thomas_safe,
    woods_saxon_safe,
)


def interaction_central(r, V, W, R, a, Zz, Rc):
    nucl = -(V + 1j * W) * woods_saxon_safe(r, R, a)
    coul = coulomb_charged_sphere(r, Zz, Rc)
    return nucl + coul


def interaction_spin_orbit(r, Vso, Wso, Rso, aso):
    return 2 * (Vso + 1j * Wso) * thomas_safe(r, Rso, aso)

In [5]:
solver = []
core_solver = jitr.rmatrix.Solver(50)
reaction = jitr.reactions.ElasticReaction(
    target=target,
    projectile=projectile,
)


# get kinematics and parameters for this experiment
kinematics = reaction.kinematics(Elab)

channel_radius_fm = a / kinematics.k

solver = jitr.xs.elastic.DifferentialWorkspace.build_from_system(
    reaction=reaction,
    kinematics=kinematics,
    channel_radius_fm=channel_radius_fm,
    solver=core_solver,
    lmax=50,
    angles=angles,
)

In [6]:
R = 1.2 * target[0] ** (1.0 / 3)
Zz = target[1] * projectile[1]
Vso, Wso = 0, 0
V, W = 50, 10

In [7]:
a_diffuse = 0.8
central_params_diffuse = (V, W, R, a_diffuse, Zz, R)
so_params_diffuse = (Vso, Wso, R, a_diffuse)

a_sharp = 0.2
central_params_sharp = (V, W, R, a_sharp, Zz, R)
so_params_sharp = (Vso, Wso, R, a_sharp)


wvp_diffuse, wvm_diffuse = solver.integral_workspace.wavefunctions(
    sgrid,
    interaction_central,
    interaction_spin_orbit,
    central_params_diffuse,
    so_params_diffuse,
)
wvp_sharp, wvm_sharp = solver.integral_workspace.wavefunctions(
    sgrid,
    interaction_central,
    interaction_spin_orbit,
    central_params_sharp,
    so_params_sharp,
)

xs_diffuse = solver.xs(
    interaction_central,
    interaction_spin_orbit,
    central_params_diffuse,
    so_params_diffuse,
)
xs_sharp = solver.xs(
    interaction_central,
    interaction_spin_orbit,
    central_params_sharp,
    so_params_sharp,
)

In [8]:
r = sgrid / kinematics.k

In [9]:
fig, axes = plt.subplots(3, 1, figsize=(6, 6))
axes[0].plot(r, interaction_central(r, *central_params_diffuse).real, color=colors[0])
axes[0].plot(
    r, interaction_central(r, *central_params_diffuse).imag, ":", color=colors[0]
)
axes[0].plot(r, interaction_central(r, *central_params_sharp).real, color=colors[1])
axes[0].plot(
    r, interaction_central(r, *central_params_sharp).imag, ":", color=colors[1]
)
#axes[0].set_xlabel(r"$r$ [fm]")
axes[0].set_ylabel(r"$U(r)$ [MeV]")
axes[0].plot([], [], "k", label=r"Re")
axes[0].plot([], [], ":k", label=r"Im")
axes[0].legend(ncol=2, framealpha=0, fontsize=12, loc="lower right")

axes[1].plot(r, wvp_diffuse[0, :].real, color=colors[0])
axes[1].plot(r, wvp_diffuse[0, :].imag, ":", color=colors[0])
axes[1].plot(r, wvp_sharp[0, :].real, color=colors[1])
axes[1].plot(r, wvp_sharp[0, :].imag, ":", color=colors[1])
axes[1].set_xlabel(r"$r$ [fm]")
axes[1].set_ylabel(r"$\langle r | s_{1/2}\rangle$ [a.u.]")

axes[2].semilogy(np.rad2deg(angles), xs_diffuse.dsdo, color=colors[0])
axes[2].semilogy(np.rad2deg(angles), xs_sharp.dsdo, color=colors[1])

axes[2].set_xlabel(r"$\theta$ [CM-degrees]")
axes[2].set_ylabel(r"$d\sigma/d\Omega$ [mb/Sr]")
plt.tight_layout()
plt.savefig("diffraction.pgf", format="pgf")